# 12 — Hospital Report Card (RQ12)

> **Pergunta:** Quais hospitais entregam resultados piores (ou melhores) do que o esperado para o perfil de pacientes que atendem?

A diferença para o nb04 (eficiência hospitalar) é que aqui usamos **ML para ajustar pelo case-mix** do paciente antes de avaliar o hospital. Um hospital que atende pacientes mais graves pode ter LOS mais longo sem ser ineficiente — o modelo controla isso.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
sys.path.insert(0, ".")
from shared import (
    load_kidney, load_cnes_enriched, load_cnes_names,
    setup_plot_style, save_plot, save_metrics, load_metrics,
    hospital_name, COLORS, DATA_DIR, PLOT_DIR,
)

setup_plot_style()
kidney = load_kidney()
cnes = load_cnes_enriched()
names_df = load_cnes_names()

kidney["sub_diag"] = kidney["DIAG_PRINC"].astype(str).str[:4]
kidney["has_comorbidity"] = (
    kidney["DIAG_SECUN"].notna()
    & (kidney["DIAG_SECUN"].astype(str).str.strip() != "")
).astype(int)
kidney["bed_specialty"] = (
    kidney["ESPEC"].astype(str).map({"01": "surgical", "03": "clinical"}).fillna("other")
)
kidney["month"] = (
    pd.to_datetime(kidney["DT_INTER"], format="%Y%m%d", errors="coerce")
    .dt.month.fillna(6).astype(int)
)
kidney["long_stay"] = (kidney["DIAS_PERM"] > 7).astype(int)

print(f"Loaded {len(kidney):,} admissions, {kidney['CNES'].nunique()} hospitals")

Loaded 206,500 admissions, 510 hospitals


## 1. Modelo de Risco do Paciente (case-mix adjustment)

Mesmo modelo do nb11 — LightGBM treinado **apenas com características do paciente**. A ideia: prever o LOS/desfechos esperados dado o perfil do paciente, SEM saber em qual hospital ele foi atendido. O resíduo (real − previsto) captura o efeito do hospital.

In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, r2_score, mean_absolute_error
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb

FEATURES = ["age", "is_male", "is_emergency", "has_comorbidity", "year", "month"]
CAT_FEATURES = ["sub_diag", "bed_specialty"]

df = kidney.copy()
for col in CAT_FEATURES:
    dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
    df = pd.concat([df, dummies], axis=1)

feature_cols = FEATURES + [
    c for c in df.columns
    if c.startswith("sub_diag_") or c.startswith("bed_specialty_")
]
X = df[feature_cols].values.astype(np.float32)
y_los = df["DIAS_PERM"].values.astype(np.float32)
y_long = df["long_stay"].values.astype(np.int32)
y_mort = df["MORTE"].values.astype(np.int32)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

los_params = dict(
    objective="regression", metric="mae", n_estimators=300,
    max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, verbose=-1, random_state=42,
)
cls_params = dict(
    objective="binary", metric="auc", n_estimators=300,
    max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, verbose=-1, random_state=42,
    is_unbalance=True,
)

pred_los = np.zeros(len(X))
pred_long = np.zeros(len(X))
pred_mort = np.zeros(len(X))

for fold, (train_idx, test_idx) in enumerate(kf.split(X, y_long)):
    rng_fold = np.random.RandomState(42 + fold)
    perm = rng_fold.permutation(len(train_idx))
    cal_size = int(len(train_idx) * 0.2)
    cal_idx = train_idx[perm[:cal_size]]
    fit_idx = train_idx[perm[cal_size:]]

    m_los = lgb.LGBMRegressor(**los_params)
    m_los.fit(X[train_idx], y_los[train_idx])
    pred_los[test_idx] = m_los.predict(X[test_idx])

    m_long = lgb.LGBMClassifier(**cls_params)
    m_long.fit(X[fit_idx], y_long[fit_idx])
    cal_p = m_long.predict_proba(X[cal_idx])[:, 1]
    iso_l = IsotonicRegression(out_of_bounds="clip")
    iso_l.fit(cal_p, y_long[cal_idx])
    pred_long[test_idx] = iso_l.transform(m_long.predict_proba(X[test_idx])[:, 1])

    m_mort = lgb.LGBMClassifier(**cls_params)
    m_mort.fit(X[fit_idx], y_mort[fit_idx])
    cal_m = m_mort.predict_proba(X[cal_idx])[:, 1]
    iso_m = IsotonicRegression(out_of_bounds="clip")
    iso_m.fit(cal_m, y_mort[cal_idx])
    pred_mort[test_idx] = iso_m.transform(m_mort.predict_proba(X[test_idx])[:, 1])

    print(f"  Fold {fold+1}/5 done")

r2 = r2_score(y_los, pred_los)
mae = mean_absolute_error(y_los, pred_los)
auc_long = roc_auc_score(y_long, pred_long)
auc_mort = roc_auc_score(y_mort, pred_mort)

print(f"\nPatient Risk Model (case-mix adjustment):")
print(f"  LOS R\u00b2={r2:.3f}  MAE={mae:.2f}d")
print(f"  Long Stay AUC={auc_long:.3f}")
print(f"  Mortality AUC={auc_mort:.3f}")

/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 1/5 done


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

  Fold 4/5 done


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 5/5 done

Patient Risk Model (case-mix adjustment):
  LOS R²=0.077  MAE=1.62d
  Long Stay AUC=0.704
  Mortality AUC=0.725


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 2. Agregação por Hospital

Cada paciente agora tem um LOS previsto (baseado no seu perfil) e um LOS real. Agregando por hospital (CNES), calculamos o gap médio. Hospitais com gap positivo atendem **pior** do que o esperado para seus pacientes; gap negativo = **melhor** que o esperado.

In [3]:
df["pred_los"] = pred_los
df["pred_long"] = pred_long
df["pred_mort"] = pred_mort

MIN_ADMISSIONS = 30

hosp_agg = df.groupby("CNES").agg(
    n=("DIAS_PERM", "count"),
    actual_los=("DIAS_PERM", "mean"),
    pred_los=("pred_los", "mean"),
    actual_long_rate=("long_stay", "mean"),
    pred_long_rate=("pred_long", "mean"),
    actual_mort_rate=("MORTE", "mean"),
    pred_mort_rate=("pred_mort", "mean"),
).reset_index()

hosp_agg = hosp_agg[hosp_agg["n"] >= MIN_ADMISSIONS].copy()

hosp_agg["gap_los"] = hosp_agg["actual_los"] - hosp_agg["pred_los"]
hosp_agg["gap_long"] = hosp_agg["actual_long_rate"] - hosp_agg["pred_long_rate"]
hosp_agg["gap_mort"] = hosp_agg["actual_mort_rate"] - hosp_agg["pred_mort_rate"]

for col in ["gap_los", "gap_long", "gap_mort"]:
    hosp_agg[f"{col}_z"] = (hosp_agg[col] - hosp_agg[col].mean()) / hosp_agg[col].std()

hosp_agg["gap_composite"] = (
    0.4 * hosp_agg["gap_los_z"]
    + 0.3 * hosp_agg["gap_long_z"]
    + 0.3 * hosp_agg["gap_mort_z"]
)

hosp_agg["hospital_name"] = hosp_agg["CNES"].apply(
    lambda c: hospital_name(c, names_df)
)

# Operational features (derived from admissions data)
is_surgical = df["proc_category"] == "SURGICAL"
is_diagnostic = df["proc_category"] == "DIAGNOSTIC"
is_migration = df["MUNIC_RES"].astype(str) != df["MUNIC_MOV"].astype(str)

ops = df.groupby("CNES").agg(
    pct_surgical=("proc_category", lambda x: (x == "SURGICAL").mean()),
    pct_diagnostic=("proc_category", lambda x: (x == "DIAGNOSTIC").mean()),
    pct_ureteroscopy=("has_new_proc", "mean"),
    pct_emergency=("is_emergency", "mean"),
    pct_comorbidity=("has_comorbidity", "mean"),
    n_unique_procs=("PROC_REA", "nunique"),
    mean_age=("age", "mean"),
).reset_index()

mig = df.copy()
mig["is_migration"] = mig["MUNIC_RES"].astype(str) != mig["MUNIC_MOV"].astype(str)
pct_mig = mig.groupby("CNES")["is_migration"].mean().reset_index()
pct_mig.columns = ["CNES", "pct_migration"]
ops = ops.merge(pct_mig, on="CNES", how="left")

hosp_agg = hosp_agg.merge(ops, on="CNES", how="left")

print(f"Hospitals with \u2265{MIN_ADMISSIONS} admissions: {len(hosp_agg)}")
print(f"Gap LOS: mean={hosp_agg['gap_los'].mean():.3f}, std={hosp_agg['gap_los'].std():.3f}")
print(f"Gap composite: mean={hosp_agg['gap_composite'].mean():.3f}, std={hosp_agg['gap_composite'].std():.2f}")
print(f"Operational features added: pct_surgical, pct_diagnostic, pct_ureteroscopy, pct_emergency, pct_migration, n_unique_procs, mean_age")

Hospitals with ≥30 admissions: 283
Gap LOS: mean=0.069, std=0.887
Gap composite: mean=-0.000, std=0.76
Operational features added: pct_surgical, pct_diagnostic, pct_ureteroscopy, pct_emergency, pct_migration, n_unique_procs, mean_age


## 3. Hospital Report Card

Ranking dos hospitais por gap composto ajustado por case-mix.

In [4]:
n_show = 15

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

worst = hosp_agg.nlargest(n_show, "gap_composite")
best = hosp_agg.nsmallest(n_show, "gap_composite")

ax = axes[0]
colors_w = [COLORS["danger"]] * n_show
y_pos = range(n_show)
bars = ax.barh(y_pos, worst["gap_composite"].values, color=colors_w, alpha=0.8)
ax.set_yticks(y_pos)
labels_w = [f"{r.hospital_name} (n={r.n})" for _, r in worst.iterrows()]
ax.set_yticklabels(labels_w, fontsize=8)
ax.set_xlabel("Gap Composto (ajustado por case-mix)")
ax.set_title(f"Top {n_show} Hospitais com PIOR Desempenho", fontweight="bold", color=COLORS["danger"])
ax.invert_yaxis()
ax.axvline(0, color="gray", linestyle="--", alpha=0.3)

ax = axes[1]
colors_b = [COLORS["success"]] * n_show
bars = ax.barh(y_pos, best["gap_composite"].values, color=colors_b, alpha=0.8)
ax.set_yticks(y_pos)
labels_b = [f"{r.hospital_name} (n={r.n})" for _, r in best.iterrows()]
ax.set_yticklabels(labels_b, fontsize=8)
ax.set_xlabel("Gap Composto (ajustado por case-mix)")
ax.set_title(f"Top {n_show} Hospitais com MELHOR Desempenho", fontweight="bold", color=COLORS["success"])
ax.invert_yaxis()
ax.axvline(0, color="gray", linestyle="--", alpha=0.3)

fig.suptitle("Hospital Report Card — Ajustado por Case-Mix do Paciente",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
save_plot(fig, "hospital_report_card", prefix="12")

print(f"\n=== TOP {n_show} PIORES ===")
cols_show = ["hospital_name", "n", "actual_los", "pred_los", "gap_los",
             "actual_mort_rate", "pred_mort_rate", "gap_composite"]
print(worst[cols_show].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print(f"\n=== TOP {n_show} MELHORES ===")
print(best[cols_show].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

  Saved: 12_hospital_report_card.png

=== TOP 15 PIORES ===
                                      hospital_name   n  actual_los  pred_los  gap_los  actual_mort_rate  pred_mort_rate  gap_composite
                      HOSP MUN DR CARMINO CARICCHIO 291       7.089     2.922    4.167             0.003           0.004          3.201
       COMPLEXO HOSPITALAR PADRE BENTO DE GUARULHOS  52       4.885     2.534    2.350             0.019           0.003          2.750
      HOSP MUN PROFESSOR DOUTOR ALIPIO CORREA NETTO 640       5.958     3.039    2.919             0.009           0.004          2.510
                     HOSPITAL MUNICIPAL BRASILANDIA 108       5.324     2.517    2.807             0.009           0.003          2.509
                               HOSPITAL SANTO AMARO 167       4.784     2.627    2.157             0.018           0.004          2.330
                                     HOSPITAL IELAR  56       5.714     3.250    2.465             0.000           0.004    

## 4. Significância Estatística dos Gaps

Nem todo gap é real — hospitais com poucos pacientes podem ter gaps por acaso. Usamos bootstrap 95% CI para identificar quais gaps são estatisticamente significativos.

In [5]:
N_BOOTSTRAP = 1000
rng = np.random.RandomState(42)

ci_results = []
for _, row in hosp_agg.iterrows():
    mask = df["CNES"] == row["CNES"]
    residuals = df.loc[mask, "DIAS_PERM"].values - df.loc[mask, "pred_los"].values
    boot_means = np.array([
        rng.choice(residuals, len(residuals), replace=True).mean()
        for _ in range(N_BOOTSTRAP)
    ])
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    ci_results.append({
        "CNES": row["CNES"],
        "gap_los": row["gap_los"],
        "ci_lo": lo,
        "ci_hi": hi,
        "significant": lo > 0 or hi < 0,
        "direction": "worse" if lo > 0 else ("better" if hi < 0 else "ns"),
    })

ci_df = pd.DataFrame(ci_results)
hosp_agg = hosp_agg.merge(ci_df[["CNES", "significant", "direction"]], on="CNES", how="left")

n_sig = ci_df["significant"].sum()
n_worse = (ci_df["direction"] == "worse").sum()
n_better = (ci_df["direction"] == "better").sum()
n_ns = (ci_df["direction"] == "ns").sum()

print(f"Gaps significativos (95% CI exclui zero): {n_sig}/{len(ci_df)} ({n_sig/len(ci_df)*100:.0f}%)")
print(f"  Significativamente piores: {n_worse}")
print(f"  Significativamente melhores: {n_better}")
print(f"  Não significativos: {n_ns}")

# Visualization
fig, ax = plt.subplots(figsize=(14, 6))
sorted_ci = ci_df.sort_values("gap_los")
colors = [COLORS["danger"] if d == "worse" else COLORS["success"] if d == "better" else "lightgray"
          for d in sorted_ci["direction"]]
ax.bar(range(len(sorted_ci)), sorted_ci["gap_los"], color=colors, alpha=0.7, width=1.0)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Hospitais (ordenados por gap)")
ax.set_ylabel("Gap LOS (dias)")
ax.set_title(f"Gaps Hospitalares Ajustados por Case-Mix\n"
             f"{n_worse} piores (vermelho), {n_better} melhores (verde), {n_ns} não significativos (cinza)",
             fontweight="bold")
ax.set_xticks([])
fig.tight_layout()
save_plot(fig, "gap_significance", prefix="12")

Gaps significativos (95% CI exclui zero): 194/283 (69%)
  Significativamente piores: 83
  Significativamente melhores: 111
  Não significativos: 89
  Saved: 12_gap_significance.png


## 5. O que Faz um Bom Hospital?

Cruzamos o gap ajustado por case-mix com características do hospital (CNES enriquecido) para identificar quais fatores estruturais estão associados a melhores resultados.

In [6]:
hosp_enr = hosp_agg.merge(cnes, on="CNES", how="left")

NUMERIC_CNES = [
    "active_committees", "total_beds", "clinical_beds", "surgical_beds",
    "icu_beds", "doctors", "surgeons", "nurses", "total_professionals",
    "doctors_per_bed", "nurses_per_bed",
    "equip_existing", "equip_in_use", "equip_utilization", "ct_scanners",
    "sus_bed_fraction", "total_beds_sus",
]

hosp_char_cols = [c for c in NUMERIC_CNES if c in hosp_enr.columns
                  and hosp_enr[c].notna().sum() > len(hosp_enr) * 0.3]

OPS_COLS = ["n", "pct_surgical", "pct_diagnostic", "pct_ureteroscopy",
            "pct_emergency", "pct_comorbidity", "n_unique_procs",
            "mean_age", "pct_migration"]
for col in OPS_COLS:
    if col in hosp_enr.columns:
        hosp_char_cols.append(col)

print(f"Hospital characteristics available: {len(hosp_char_cols)}")
print(f"Hospitals with CNES data: {hosp_enr[hosp_char_cols[0]].notna().sum()}/{len(hosp_enr)}")

correlations = []
for col in hosp_char_cols:
    valid = hosp_enr[[col, "gap_composite"]].dropna()
    if len(valid) > 10:
        rho, p = stats.spearmanr(valid[col], valid["gap_composite"])
        correlations.append({"feature": col, "rho": rho, "p": p, "n": len(valid)})

corr_df = pd.DataFrame(correlations).sort_values("rho")

print(f"\nTop correlates of case-mix adjusted gap:")
top_corr = pd.concat([corr_df.head(10), corr_df.tail(10)]) if len(corr_df) > 20 else corr_df
for _, row in corr_df.iterrows():
    if row["p"] < 0.05:
        sig = "***" if row["p"] < 0.001 else "**" if row["p"] < 0.01 else "*"
        print(f"  {row['feature']:30s} \u03c1={row['rho']:+.3f}  p={row['p']:.4f} {sig}")

Hospital characteristics available: 20
Hospitals with CNES data: 281/283

Top correlates of case-mix adjusted gap:
  mean_age                       ρ=-0.207  p=0.0004 ***
  pct_surgical                   ρ=-0.139  p=0.0195 *
  pct_ureteroscopy               ρ=-0.125  p=0.0349 *
  pct_emergency                  ρ=+0.165  p=0.0053 **
  sus_bed_fraction               ρ=+0.186  p=0.0018 **
  pct_diagnostic                 ρ=+0.220  p=0.0002 ***
  active_committees              ρ=+0.234  p=0.0001 ***
  equip_in_use                   ρ=+0.369  p=0.0000 ***
  total_beds                     ρ=+0.371  p=0.0000 ***
  equip_existing                 ρ=+0.373  p=0.0000 ***
  total_beds_sus                 ρ=+0.406  p=0.0000 ***


/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_30810/2064574428.py:28: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(valid[col], valid["gap_composite"])
/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_30810/2064574428.py:28: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(valid[col], valid["gap_composite"])
/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_30810/2064574428.py:28: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(valid[col], valid["gap_composite"])
/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_30810/2064574428.py:28: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(valid[col], valid["gap_composite"])


## 6. Modelo de Segundo Estágio: Gap ~ Características Hospitalares

Treinamos um LightGBM de segundo estágio para prever o gap composto a partir de características do hospital. SHAP revela quais fatores hospitalares mais importam para desempenho ajustado por case-mix.

In [7]:
import shap

valid_hosp = hosp_enr[hosp_char_cols + ["gap_composite"]].dropna()
X_hosp = valid_hosp[hosp_char_cols].values.astype(np.float32)
y_gap = valid_hosp["gap_composite"].values.astype(np.float32)

hosp_model = lgb.LGBMRegressor(
    objective="regression", metric="mae", n_estimators=200,
    max_depth=4, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, verbose=-1, random_state=42,
    min_child_samples=5,
)
hosp_model.fit(X_hosp, y_gap)

from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(hosp_model, X_hosp, y_gap, cv=5, scoring="r2")
r2_hosp = cv_scores.mean()
print(f"Second-stage model (gap ~ hospital features):")
print(f"  R\u00b2 (5-fold CV): {r2_hosp:.3f} \u00b1 {cv_scores.std():.3f}")
print(f"  Training R\u00b2: {r2_score(y_gap, hosp_model.predict(X_hosp)):.3f}")

explainer = shap.TreeExplainer(hosp_model)
shap_values = explainer.shap_values(X_hosp)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
importance = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(importance)[-10:]
ax.barh(range(len(top_idx)), importance[top_idx], color=COLORS["primary"], alpha=0.8)
ax.set_yticks(range(len(top_idx)))
ax.set_yticklabels([hosp_char_cols[i] for i in top_idx], fontsize=9)
ax.set_xlabel("Mean |SHAP|")
ax.set_title("Feature Importance (SHAP)", fontweight="bold")

ax = axes[1]
shap.summary_plot(shap_values, X_hosp, feature_names=hosp_char_cols,
                  max_display=10, show=False, plot_size=None)
plt.sca(ax)

fig.suptitle("O que Faz um Bom Hospital? (Modelo de Segundo Estágio)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
save_plot(fig, "hospital_shap", prefix="12")

/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Second-stage model (gap ~ hospital features):
  R² (5-fold CV): 0.155 ± 0.162
  Training R²: 0.952


  Saved: 12_hospital_shap.png


## 7. Validação Cruzada com Eficiência do nb04

O gap ajustado por case-mix deve correlacionar com o score de eficiência do nb04. Se ambas as abordagens concordam, a evidência é mais robusta.

In [8]:
try:
    nb04_metrics = load_metrics("04_hospital_efficiency")
    print("Loaded nb04 metrics")
except Exception:
    nb04_metrics = None
    print("nb04 metrics not available")

hosp_stats_simple = df.groupby("CNES").agg(
    h_los=("DIAS_PERM", "mean"),
    h_n=("DIAS_PERM", "count"),
).reset_index()
hosp_stats_simple = hosp_stats_simple[hosp_stats_simple["h_n"] >= 10]
p75 = hosp_stats_simple["h_los"].quantile(0.75)
hosp_stats_simple["efficiency"] = 1 - (hosp_stats_simple["h_los"] / p75).clip(0, 1)

hosp_compare = hosp_agg.merge(hosp_stats_simple[["CNES", "efficiency"]], on="CNES", how="left")

valid = hosp_compare.dropna(subset=["efficiency", "gap_composite"])
rho_eff, p_eff = stats.spearmanr(valid["gap_composite"], valid["efficiency"])

print(f"\nCorrelation: case-mix adjusted gap vs simple efficiency")
print(f"  Spearman \u03c1 = {rho_eff:+.3f}, p = {p_eff:.6f}")
print(f"  N hospitals = {len(valid)}")
print(f"  Direction: {'COERENTE' if rho_eff < 0 else 'INESPERADO'} (expected negative)")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(valid["efficiency"], valid["gap_composite"],
           s=valid["n"]/3, alpha=0.5, color=COLORS["primary"])
ax.set_xlabel("Efici\u00eancia Simples (nb04)")
ax.set_ylabel("Gap Composto (ajustado por case-mix)")
ax.set_title(f"Case-Mix Adjusted Gap vs Simple Efficiency\n"
             f"\u03c1={rho_eff:+.3f}, p={p_eff:.4f}", fontweight="bold")
ax.axhline(0, color="gray", linestyle="--", alpha=0.3)

z = np.polyfit(valid["efficiency"], valid["gap_composite"], 1)
x_line = np.linspace(valid["efficiency"].min(), valid["efficiency"].max(), 100)
ax.plot(x_line, np.polyval(z, x_line), color=COLORS["danger"], linestyle="--", alpha=0.7)

fig.tight_layout()
save_plot(fig, "gap_vs_efficiency", prefix="12")

Loaded nb04 metrics

Correlation: case-mix adjusted gap vs simple efficiency
  Spearman ρ = -0.836, p = 0.000000
  N hospitals = 283
  Direction: COERENTE (expected negative)
  Saved: 12_gap_vs_efficiency.png


## 8. Report Card Detalhado

Tabela completa com o veredicto por hospital: gap significativo, direção, e comparação com eficiência simples.

In [9]:
report_card = hosp_agg.merge(
    hosp_stats_simple[["CNES", "efficiency"]], on="CNES", how="left"
).sort_values("gap_composite", ascending=False)

report_card["verdict"] = report_card.apply(
    lambda r: "\u26a0\ufe0f REVISÃO" if r.get("significant") and r["gap_composite"] > 0.5
    else "\u2705 REFERÊNCIA" if r.get("significant") and r["gap_composite"] < -0.5
    else "\u2796 OK", axis=1
)

n_review = (report_card["verdict"] == "\u26a0\ufe0f REVISÃO").sum()
n_reference = (report_card["verdict"] == "\u2705 REFERÊNCIA").sum()
n_ok = (report_card["verdict"] == "\u2796 OK").sum()

print(f"Hospital Report Card Summary:")
print(f"  \u26a0\ufe0f Precisa revisão:   {n_review}")
print(f"  \u2705 Centro de referência: {n_reference}")
print(f"  \u2796 Desempenho esperado:  {n_ok}")

report_card.to_csv(DATA_DIR / "hospital_report_card.csv", index=False)
print(f"\nSaved report card: {len(report_card)} hospitals")

print(f"\n=== HOSPITAIS QUE PRECISAM REVIS\u00c3O ===")
review = report_card[report_card["verdict"] == "\u26a0\ufe0f REVISÃO"]
cols = ["hospital_name", "n", "actual_los", "pred_los", "gap_los",
        "actual_mort_rate", "gap_composite", "efficiency"]
print(review[cols].head(20).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print(f"\n=== CENTROS DE REFER\u00caNCIA ===")
ref = report_card[report_card["verdict"] == "\u2705 REFERÊNCIA"]
print(ref[cols].head(20).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Hospital Report Card Summary:
  ⚠️ Precisa revisão:   49
  ✅ Centro de referência: 66
  ➖ Desempenho esperado:  168

Saved report card: 283 hospitals

=== HOSPITAIS QUE PRECISAM REVISÃO ===
                                             hospital_name   n  actual_los  pred_los  gap_los  actual_mort_rate  gap_composite  efficiency
                             HOSP MUN DR CARMINO CARICCHIO 291       7.089     2.922    4.167             0.003          3.201       0.000
              COMPLEXO HOSPITALAR PADRE BENTO DE GUARULHOS  52       4.885     2.534    2.350             0.019          2.750       0.000
             HOSP MUN PROFESSOR DOUTOR ALIPIO CORREA NETTO 640       5.958     3.039    2.919             0.009          2.510       0.000
                            HOSPITAL MUNICIPAL BRASILANDIA 108       5.324     2.517    2.807             0.009          2.509       0.000
                                      HOSPITAL SANTO AMARO 167       4.784     2.627    2.157             0.018    

## 9. Veredicto da Validação

In [10]:
checks = [
    ("Gaps significativos (>20% dos hospitais)",
     n_sig / len(ci_df) > 0.20),
    ("Correlação gap vs eficiência (|\u03c1| > 0.3)",
     abs(rho_eff) > 0.3),
    ("Modelo 2\u00ba estágio R\u00b2 > 0.15",
     r2_hosp > 0.15 if r2_hosp else False),
    ("Hospital ranking \u2265 10 para revisão",
     n_review >= 10),
]

print("=" * 65)
print("  VEREDICTO DA VALIDAÇÃO — HOSPITAL REPORT CARD")
print("=" * 65)
n_pass = 0
for label, passed in checks:
    status = "\u2705 PASS" if passed else "\u274c FAIL"
    print(f"  {status} | {label}")
    if passed:
        n_pass += 1
print(f"\n  Resultado: {n_pass}/{len(checks)} testes passaram")
if n_pass >= 3:
    print("  \u2192 REPORT CARD VALIDADO")
else:
    print("  \u2192 REPORT CARD PARCIALMENTE VALIDADO")
print("=" * 65)

metrics = {
    "total_patients": int(len(kidney)),
    "total_hospitals": int(kidney["CNES"].nunique()),
    "hospitals_analyzed": int(len(hosp_agg)),
    "min_admissions": MIN_ADMISSIONS,
    "patient_model_r2": round(r2, 3),
    "patient_model_mae": round(mae, 2),
    "patient_model_auc_long": round(auc_long, 3),
    "patient_model_auc_mort": round(auc_mort, 3),
    "hospitals_significant_gap": int(n_sig),
    "hospitals_worse": int(n_worse),
    "hospitals_better": int(n_better),
    "pct_significant": round(n_sig / len(ci_df) * 100, 1),
    "gap_efficiency_rho": round(rho_eff, 3),
    "gap_efficiency_p": round(p_eff, 6),
    "second_stage_r2": round(r2_hosp, 3) if r2_hosp else None,
    "n_review": int(n_review),
    "n_reference": int(n_reference),
    "validation_pass": int(n_pass),
    "validation_total": int(len(checks)),
}
save_metrics(metrics, "12_hospital_report_card")

  VEREDICTO DA VALIDAÇÃO — HOSPITAL REPORT CARD
  ✅ PASS | Gaps significativos (>20% dos hospitais)
  ✅ PASS | Correlação gap vs eficiência (|ρ| > 0.3)
  ✅ PASS | Modelo 2º estágio R² > 0.15
  ✅ PASS | Hospital ranking ≥ 10 para revisão

  Resultado: 4/4 testes passaram
  → REPORT CARD VALIDADO
  Saved metrics: 12_hospital_report_card.json
